In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import clip
from torch.nn import functional as F
import torch.nn as nn
from torchvision import transforms
from PIL import Image
train = False
classes = None
pictures= None

def load_data():
    data_list = []
    label_list = []
    texts = []
    images = []
    
    if train:

        text_directory ="/THINGS-Data/THINGS-EEG_images_set/training_images"
    else:

        text_directory ="/THINGS-Data/THINGS-EEG_images_set/test_images"

    dirnames = [d for d in os.listdir(text_directory) if os.path.isdir(os.path.join(text_directory, d))]
    dirnames.sort()
    
    if classes is not None:
        dirnames = [dirnames[i] for i in classes]

    for dir in dirnames:

        try:
            idx = dir.index('_')
            description = dir[idx+1:]
        except ValueError:
            print(f"Skipped: {dir} due to no '_' found.")
            continue
            
        new_description = f"{description}"
        texts.append(new_description)

    if train:
        
        img_directory ="/THINGS-Data/THINGS-EEG_images_set/training_images"
        
    else:

        img_directory ="/THINGS-Data/THINGS-EEG_images_set/test_images"
    
    all_folders = [d for d in os.listdir(img_directory) if os.path.isdir(os.path.join(img_directory, d))]
    all_folders.sort()

    if classes is not None and pictures is not None:
        images = []
        for i in range(len(classes)):
            class_idx = classes[i]
            pic_idx = pictures[i]
            if class_idx < len(all_folders):
                folder = all_folders[class_idx]
                folder_path = os.path.join(img_directory, folder)
                all_images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
                all_images.sort()
                if pic_idx < len(all_images):
                    images.append(os.path.join(folder_path, all_images[pic_idx]))
    elif classes is not None and pictures is None:
        images = []
        for i in range(len(classes)):
            class_idx = classes[i]
            if class_idx < len(all_folders):
                folder = all_folders[class_idx]
                folder_path = os.path.join(img_directory, folder)
                all_images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
                all_images.sort()
                images.extend(os.path.join(folder_path, img) for img in all_images)
    elif classes is None:
        images = []
        for folder in all_folders:
            folder_path = os.path.join(img_directory, folder)
            all_images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
            all_images.sort()  
            images.extend(os.path.join(folder_path, img) for img in all_images)
    else:

        print("Error")
    return texts, images
texts, images = load_data()
# images

In [ ]:
texts , images
print('test text  len', len(texts))#200 如 'aircraft_carrier',
print('test images len', len(images))#200 如'/THINGS-Data/THINGS-EEG_images_set/test_images/00001_aircraft_carrier/aircraft_carrier_06s.jpg'


test text  len 200
test images len 200


In [ ]:
import os

import torch
import torch.optim as optim
import argparse  

try:
    Tensor = torch.Tensor
except Exception:
    Tensor = object

import datetime  # 
import math  # 

from braindecode.models import EEGNetv4, ATCNet, EEGConformer, EEGITNet, ShallowFBCSPNet
from torch.nn import CrossEntropyLoss
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ["WANDB_API_KEY"] = "KEY"  
os.environ["WANDB_MODE"] = 'offline'
from itertools import combinations

import clip
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torchvision.transforms as transforms
import tqdm
# from BrainAligning_retrieval.eegdatasets_leaveone import EEGDataset
from eegdatasets_leaveone import EEGDataset
from einops.layers.torch import Rearrange, Reduce
from lavis.models.clip_models.loss import ClipLoss
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss
import torch
import torch.nn as nn
import torch.nn.functional as F

class SigLIPLoss(nn.Module):

    def __init__(self, local_loss=False, gather_with_grad=False, cache_labels=False, rank=0, world_size=1, use_horovod=False):
        super().__init__()
        # keep signature similar to lavis ClipLoss for easy swap
        self.local_loss = local_loss
        self.gather_with_grad = gather_with_grad
        self.cache_labels = cache_labels
        self.rank = rank
        self.world_size = world_size
        self.use_horovod = use_horovod

        self.prev_num_logits = 0
        self.labels = {}

    def forward(self, image_features, text_features, logit_scale):
        """
        image_features, text_features: (N, D)
        logit_scale: nn.Parameter (usually log-scale), will use exp()
        """
        device = image_features.device

        # L2 normalize features (important)
        image_features = F.normalize(image_features, dim=1)
        text_features = F.normalize(text_features, dim=1)

        # common CLIP uses scale = exp(logit_scale)
        try:
            scale = logit_scale.exp()
        except Exception:
            # if logit_scale is float tensor without .exp()
            scale = torch.exp(logit_scale)

        # single-gpu simple case (no distributed gather implemented here)
        logits_per_image = scale * image_features @ text_features.T
        logits_per_text = logits_per_image.T

        num_logits = logits_per_image.shape[0]
        # prepare labels (0..N-1)
        if self.prev_num_logits != num_logits or device not in self.labels:
            labels = torch.arange(num_logits, device=device, dtype=torch.long)
            if self.world_size > 1 and self.local_loss:
                labels = labels + num_logits * self.rank
            if self.cache_labels:
                self.labels[device] = labels
                self.prev_num_logits = num_logits
        else:
            labels = self.labels[device]

        loss = (F.cross_entropy(logits_per_image, labels) + F.cross_entropy(logits_per_text, labels)) / 2
        return loss

def barlow_twins_loss(z, lambd_offdiag=0.005, eps=1e-9):
    # z: (N, D)
    N, D = z.shape
    # normalize feature dims (batch-wise)
    z_norm = (z - z.mean(dim=0, keepdim=True)) / (z.std(dim=0, keepdim=True) + eps)
    c = (z_norm.t() @ z_norm) / N  # (D, D) cross-correlation
    on_diag = torch.diagonal(c).add(-1).pow(2).sum()
    off_diag = c - torch.diag(torch.diag(c))
    off_diag_loss = off_diag.pow(2).sum()
    loss = on_diag + lambd_offdiag * off_diag_loss
    return loss

from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Dataset
import random
# from utils import wandb_logger
from util import wandb_logger
import csv



class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
       
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        
        div_term = torch.exp(torch.arange(0, d_model + 1, 2).float() * (-math.log(10000.0) / d_model))
        
        
        pe[:, 0::2] = torch.sin(position * div_term[:d_model // 2 + 1])
        pe[:, 1::2] = torch.cos(position * div_term[:d_model // 2])

        
        self.register_buffer('pe', pe)

    def forward(self, x):
       
        pe = self.pe[:x.size(0), :].unsqueeze(1).repeat(1, x.size(1), 1)
        #pe shape =torch.Size([250, bs, 63])
        x = x + pe

        return x


class EEGAttention(nn.Module):
    def __init__(self, channel, d_model, nhead):
        super(EEGAttention, self).__init__()
        # 初始化位置编码器
        self.pos_encoder = PositionalEncoding(d_model)
        # 创建Transformer编码器层
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        # 创建完整的Transformer编码器
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=1)
        self.channel = channel
        self.d_model = d_model

    def forward(self, src):
        # 调整输入张量的维度顺序以适应Transformer
        src = src.permute(2, 0, 1)  # 
        src = self.pos_encoder(src) 
        
        output = self.transformer_encoder(src)#output shape =torch.Size([250, bs, 63])
        
        # 恢复原始的维度顺序
        return output.permute(1, 2, 0)  #

# 定义补丁嵌入类
class PatchEmbedding(nn.Module):
    def __init__(self, emb_size=40):
        super().__init__()
        # 设置输入数据的形状
        self.shape = (63, 250)
        # 使用EEGNetv4作为时间序列卷积网络
        self.tsconv = EEGNetv4(
            in_chans=self.shape[0],
            n_classes=1440,   
            input_window_samples=self.shape[1],
            final_conv_length='auto',
            pool_mode='mean',
            F1=8,
            D=20,
            F2=160,
            kernel_length=4,
            third_kernel_size=(4, 2),
            drop_prob=0.25
        )

    def forward(self, x: Tensor) -> Tensor:
        x = x.unsqueeze(3)     
        # print("x", x.shape)   # shape =torch.Size([bs, 63, 250, 1])
        x = self.tsconv(x)
        # After tsconv shape =torch.Size([1024, 1440])
        return x

class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x

class FlattenHead(nn.Sequential):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        
        x = x.contiguous().view(x.size(0), -1)
        return x

# 定义EEG编码器类
class Enc_eeg(nn.Sequential):
    def __init__(self, emb_size=40, **kwargs):
        super().__init__(
            PatchEmbedding(emb_size),
            FlattenHead()
        )


class GEGLU(nn.Module):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.proj = nn.Linear(dim_in, dim_out * 2)

    def forward(self, x):
        x, gate = self.proj(x).chunk(2, dim=-1)
        return x * F.gelu(gate)
    
class Proj_eeg(nn.Sequential):
    def __init__(self, embedding_dim=1440, proj_dim=1024, drop_proj=0.5):
        super().__init__(

            GEGLU(embedding_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.Dropout(drop_proj),
            
     
            ResidualAdd(nn.Sequential(
                GEGLU(proj_dim, proj_dim), 
                nn.Dropout(drop_proj),
            )),
            
        
            ResidualAdd(nn.Sequential(
                nn.Linear(proj_dim, proj_dim),
                nn.GELU(), 
                nn.Dropout(drop_proj),
            )),
            
            nn.LayerNorm(proj_dim),
        )


import torch.nn.functional as F
class DynamicResGNNLayer(nn.Module):
    def __init__(self, channels, alpha=0.2):
        super().__init__()
        self.fc = nn.Linear(channels, channels)
        self.alpha = alpha  # 残差权重
    def forward(self, x):

        x_mean = x.mean(dim=2)  #
        adj = torch.matmul(x_mean, x_mean.transpose(1, 0))  
        adj = torch.matmul(x_mean.unsqueeze(2), x_mean.unsqueeze(1))
        adj = F.softmax(adj, dim=-1) 

        # 
        x_gcn = torch.matmul(adj, x)  # [batch, channels, channels] x [batch, channels, seq_len] -> [batch, channels, seq_len]
        x_gcn = self.fc(x_gcn.transpose(1,2)).transpose(1,2)  # [batch, channels, seq_len]

       
        out = self.alpha * x + (1 - self.alpha) * x_gcn
        return out

import torch
import torch.nn as nn
import torch.nn.functional as F

class PrototypeRefinementLayer(nn.Module):
    """
    Prototype-based EEG Representation Enhancement Module
    Maintains a learnable prototype bank to de-noise EEG embeddings via soft-clustering/attention mechanism.
    """
    def __init__(self, embed_dim=1024, num_prototypes=64, temperature=0.07, residual_alpha=0.3):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_prototypes = num_prototypes
        self.temperature = temperature
        self.residual_alpha = residual_alpha 

        self.prototypes = nn.Parameter(torch.randn(num_prototypes, embed_dim))
        
        self.norm = nn.LayerNorm(embed_dim)

    def init_prototypes(self, visual_features):
        try:
            from sklearn.cluster import KMeans
            if isinstance(visual_features, torch.Tensor):
                features_np = visual_features.detach().cpu().numpy()
            else:
                features_np = visual_features
            
            print(f"Initializing {self.num_prototypes} prototypes via K-means...")
            kmeans = KMeans(n_clusters=self.num_prototypes, n_init='auto', random_state=42)
            kmeans.fit(features_np)
            centers = torch.from_numpy(kmeans.cluster_centers_).float()
            
            with torch.no_grad():
                self.prototypes.copy_(centers)
                self.prototypes.data = F.normalize(self.prototypes.data, dim=-1)
            print("Prototype initialization done.")
        except ImportError:
            print("sklearn not found or error in K-means, skipping prototype initialization.")

    def forward(self, x):
        q = F.normalize(x, dim=-1) 
        k = F.normalize(self.prototypes, dim=-1) 

        sim = torch.matmul(q, k.t()) / self.temperature

        attn_weights = F.softmax(sim, dim=-1) 

        x_proto = torch.matmul(attn_weights, self.prototypes)

        out = (1 - self.residual_alpha) * x + self.residual_alpha * x_proto
        
        out = self.norm(out)
        
        return out

class ATM_E(nn.Module):    
    def __init__(self, num_channels=63, sequence_length=250, num_subjects=1, num_features=64, num_latents=1024, num_blocks=1):
        super(ATM_E, self).__init__()
        
        self.attention_model = EEGAttention(num_channels, num_channels, nhead=1)   
        
        self.subject_wise_linear = nn.ModuleList([nn.Linear(sequence_length, sequence_length) for _ in range(num_subjects)])
       
        self.enc_eeg = Enc_eeg()
        self.proj_eeg = Proj_eeg()        
        
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        
        self.loss_func = SigLIPLoss()
        
        self.dynamic_gnn = DynamicResGNNLayer(num_channels)
        
        
        self.proto_refine = PrototypeRefinementLayer(embed_dim=num_latents, num_prototypes=16, temperature=0.07, residual_alpha=0.2)
         
    def forward(self, x):
        x = self.dynamic_gnn(x)
        
        x = self.attention_model(x)
        

        

        x = self.subject_wise_linear[0](x)

      
        eeg_embedding = self.enc_eeg(x)
        out = self.proj_eeg(eeg_embedding)
        
        out = self.proto_refine(out)

        return out  

import argparse 
import datetime  
import math 
import re  
def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None



    

def get_eegfeatures(sub, eegmodel, dataloader, device, text_features_all, img_features_all, k):
    eegmodel.eval()# 
    text_features_all = text_features_all.to(device).float()
    img_features_all = img_features_all.to(device).float()
    total_loss = 0  # 
    correct = 0     # 
    total = 0    #
    alpha =0.9    # 
    top5_correct = 0
    top5_correct_count = 0

    all_labels = set(range(text_features_all.size(0)))
    top5_acc = 0
    mse_loss_fn = nn.MSELoss() # MSE损失函数
    ridge_lambda = 0.1
    save_features = True
    features_list = []  # List to store features    # 存储特征的列表
    with torch.no_grad(): # 不计算梯度
        for batch_idx, (eeg_data, labels, text, text_features, img, img_features) in enumerate(dataloader):
            eeg_data = eeg_data.to(device)
            text_features = text_features.to(device).float()
            labels = labels.to(device)
            img_features = img_features.to(device).float()
            
            batch_size = eeg_data.size(0)  # Assume the first element is the data tensor
            subject_id = extract_id_from_string(sub)
            # eeg_data = eeg_data.permute(0, 2, 1)
            subject_ids = torch.full((batch_size,), subject_id, dtype=torch.long).to(device)
            # if not config.insubject:
            #     subject_ids = torch.full((batch_size,), -1, dtype=torch.long).to(device)          
            eeg_features = eeg_model(eeg_data)  # 通过EEG模型
            # ----------- 新增：保存当前 batch 的特征到列表 -----------
            # detach 并移到 cpu，避免占用显存和梯度信息
            features_list.append(eeg_features.detach().cpu())
            # ----------------------------------------------------
        
            logit_scale = eeg_model.logit_scale 
                   
            regress_loss =  mse_loss_fn(eeg_features, img_features)
            # print("eeg_features", eeg_features.shape)
            # print(torch.std(eeg_features, dim=-1))
            # print(torch.std(img_features, dim=-1))
            # l2_norm = sum(p.pow(2.0).sum() for p in model.parameters())
            # loss = (regress_loss + ridge_lambda * l2_norm)       
            img_loss = eegmodel.loss_func(eeg_features, img_features, logit_scale)
            text_loss = eegmodel.loss_func(eeg_features, text_features, logit_scale)
            contrastive_loss = img_loss# 对比损失
            # loss = img_loss + text_loss

            regress_loss =  mse_loss_fn(eeg_features, img_features)
            # print("text_loss", text_loss)
            # print("img_loss", img_loss)
            # print("regress_loss", regress_loss)            
            # l2_norm = sum(p.pow(2.0).sum() for p in model.parameters())
            # loss = (regress_loss + ridge_lambda * l2_norm)     
            # #
            loss = alpha * regress_loss *10 + (1 - alpha) * contrastive_loss*10
            # print("loss", loss)
            total_loss += loss.item()
            #
            for idx, label in enumerate(labels):

                possible_classes = list(all_labels - {label.item()})
                selected_classes = random.sample(possible_classes, k-1) + [label.item()]
                selected_img_features = img_features_all[selected_classes]
                

                logits_img = logit_scale * eeg_features[idx] @ selected_img_features.T
                # logits_text = logit_scale * eeg_features[idx] @ selected_text_features.T
                # logits_single = (logits_text + logits_img) / 2.0
                logits_single = logits_img
                # print("logits_single", logits_single.shape)
                # 
                # predicted_label = selected_classes[torch.argmax(logits_single).item()]
                predicted_label = selected_classes[torch.argmax(logits_single).item()] # (n_batch, ) \in {0, 1, ..., n_cls-1}
                if predicted_label == label.item():
                    correct += 1        
                total += 1
        # ---
        if save_features:
            if len(features_list) > 0:
                features_tensor = torch.cat(features_list, dim=0)
                print("features_tensor", features_tensor.shape,f"save to NotebookGCN_S_eeg_features_{sub}.pt")
                torch.save(features_tensor.cpu(), f"NotebookGCN_S_eeg_features_{sub}.pt")

            else:
                # 
                features_tensor = torch.empty((0, img_features_all.size(1)))
                print("features_tensor is empty, skipped saving.")
        # ------------------------------------------------------------------
        # if save_features:
        #     features_tensor = torch.cat(features_list, dim=0)
        #     print("features_tensor", features_tensor.shape)
        #     torch.save(features_tensor.cpu(), f"ATM_S_eeg_features_{sub}.pt")  # Save features as .pt file
    average_loss = total_loss / (batch_idx+1)
    accuracy = correct / total
    return average_loss, accuracy, labels, features_tensor.cpu()
# 配置参数
from IPython.display import Image, display
config = {
"data_path": "/EEG_Image_decode/THINGS-Data/EEG/Preprocessed_data_250Hz",
"project": "train_pos_img_text_rep",
"entity": "sustech_rethinkingbci",
"name": "lr=3e-4_img_pos_pro_eeg",
"lr": 3e-4,
"epochs": 50,
"batch_size": 1024,
"logger": True,
"encoder_type":'ATMS',
}
# 设置设备
device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")

data_path = config['data_path']

emb_img_test = torch.load('/EEG_Image_decode/Generation/ViT-H-14_features_test.pt')#只包含图片的文本和image特征 不包含eeg特征
emb_img_train = torch.load('/EEG_Image_decode/Generation/ViT-H-14_features_train.pt')
print("emb_img_test", emb_img_test['img_features'].shape)
print("emb_img_train", emb_img_train['img_features'].shape)
# emb_img_test torch.Size([200, 1024])
# emb_img_train torch.Size([16540, 1024])

# 创建EEG模型
eeg_model = ATM_E()
print('number of parameters:', sum([p.numel() for p in eeg_model.parameters()]))



eeg_model.load_state_dict(torch.load("/EEG_Image_decode/Generation/models/BestfirstGNN_notebook_woNSR/sub-08_79_best.pth"))

eeg_model = eeg_model.to(device)
sub = 'sub-08'


In [ ]:
import os

import torch
import torch.optim as optim
from torch.nn import CrossEntropyLoss
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader

os.environ["WANDB_API_KEY"] = "KEY"
os.environ["WANDB_MODE"] = 'offline'
from itertools import combinations

import clip
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torchvision.transforms as transforms
import tqdm
from eegdatasets_leaveone import EEGDataset

from einops.layers.torch import Rearrange, Reduce

from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Dataset
import random
from util import wandb_logger
from braindecode.models import EEGNetv4, ATCNet, EEGConformer, EEGITNet, ShallowFBCSPNet
import csv
from torch import Tensor
import itertools
import math
import re
from subject_layers.Transformer_EncDec import Encoder, EncoderLayer
from subject_layers.SelfAttention_Family import FullAttention, AttentionLayer
from subject_layers.Embed import DataEmbedding
import numpy as np
from loss import ClipLoss
import argparse
from torch import nn
from torch.optim import AdamW
#####################################################################################
# 创建测试数据集和数据加载器
test_dataset = EEGDataset(data_path, subjects= [sub], train=False)
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=0)
text_features_test_all = test_dataset.text_features
img_features_test_all = test_dataset.img_features
# 获取测试集的EEG特征
test_loss, test_accuracy,labels, eeg_features_test = get_eegfeatures(sub, eeg_model, test_loader, device, text_features_test_all, img_features_test_all,k=200)
print(f" - Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

In [ ]:
#训练集eegEmbedding 
####################################################################################
train_dataset = EEGDataset(data_path, subjects= [sub], train=True)
train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=0)
text_features_test_all = train_dataset.text_features
img_features_test_all = train_dataset.img_features

train_loss, train_accuracy, labels, eeg_features_train = get_eegfeatures(sub, eeg_model, train_loader, device, text_features_test_all, img_features_test_all,k=200)
print(f" - Test Loss: {train_loss:.4f}, Test Accuracy: {train_accuracy:.4f}")
#####################################################################################

# 训练集 data_tensor torch.Size([66160, 63, 250]) 16540*4次重复
# Data tensor shape: torch.Size([66160, 63, 250]), label tensor shape: torch.Size([66160]), text length: 1654, image length: 16540
# features_tensor torch.Size([66160, 1024]) save to ATM_S_eeg_features_sub-08.pt

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import open_clip
from matplotlib.font_manager import FontProperties

import importlib
try:
    import huggingface_hub as _hf
    # 如果缺少旧 API，就用现有的 hf_hub_download 作为替代
    if not hasattr(_hf, "cached_download") and hasattr(_hf, "hf_hub_download"):
        _hf.cached_download = _hf.hf_hub_download
    if not hasattr(_hf, "cached_path") and hasattr(_hf, "hf_hub_download"):
        # 有些库可能调用 cached_path
        _hf.cached_path = _hf.hf_hub_download
except Exception:
    pass


import sys
from diffusion_prior import *
from custom_pipeline import *
# os.environ["CUDA_VISIBLE_DEVICES"] = "5" 
device = torch.device('cuda:7' if torch.cuda.is_available() else 'cpu')

In [ ]:
emb_img_train_4 = emb_img_train['img_features'].view(1654,10,1,1024).repeat(1,1,4,1).view(-1,1024)


emb_eeg      = torch.load('/EEG_Image_decode/Generation/NotebookGCN_S_eeg_features_sub-08.pt')
emb_eeg_test = torch.load('/EEG_Image_decode/Generation/NotebookGCN_S_eeg_features_sub-08_test.pt')


In [8]:
emb_eeg.shape, emb_eeg_test.shape

(torch.Size([66160, 1024]), torch.Size([200, 1024]))

In [9]:
eeg_features_train=emb_eeg # eeg_features_train.shape#torch.Size([66160, 1024])
eeg_features_train

# emb_img_train_4.shape           #torch.Size([66160, 1024])

tensor([[-0.1096,  0.6423,  0.5289,  ...,  0.7031, -0.2648, -0.0788],
        [ 0.3627,  0.3171,  1.0784,  ...,  0.5679,  0.3042, -0.7965],
        [-0.4813, -1.1802, -0.2075,  ...,  0.1182, -0.2297, -0.0308],
        ...,
        [-0.4746,  0.1872,  0.4882,  ..., -2.0143, -1.4183, -0.7303],
        [-0.8229,  0.4709, -0.1039,  ..., -0.5539, -0.2353, -0.1120],
        [ 0.3051,  0.2036, -0.7615,  ...,  0.2831, -1.4225, -0.8168]])

In [ ]:
# 创建嵌入数据集对象，用于训练扩散模型
# c_embeddings: 条件嵌入（EEG特征），h_embeddings: 目标嵌入（图像特征）
dataset = EmbeddingDataset(
    c_embeddings=eeg_features_train, h_embeddings=emb_img_train_4, 
    # h_embeds_uncond=h_embeds_imgnet
)

dl = DataLoader(dataset, batch_size=1024, shuffle=True, num_workers=64)

diffusion_prior = DiffusionPriorUNet(cond_dim=1024, dropout=0.1)
# number of parameters

# 打印模型中需要梯度计算的参数总数（即可训练参数数量）
print(sum(p.numel() for p in diffusion_prior.parameters() if p.requires_grad))
# 创建管道对象，封装扩散先验模型和设备信息
pipe = Pipe(diffusion_prior, device=device)

# load pretrained model# 设置模型名称
model_name = 'diffusion_prior' # 'diffusion_prior_vice_pre_imagenet' or 'diffusion_prior_vice_pre'

#训练25分钟左右 
# 训练模型

#训练部分注释掉了
###################### 需要训练请解开注释
pipe.train(dl, num_epochs=100, learning_rate=1e-3) # to 0.142 


In [ ]:
# ...existing code...
import os
# A. 禁用系统代理并强制离线（推荐，避免尝试连 hf）
for k in ("http_proxy","https_proxy","HTTP_PROXY","HTTPS_PROXY"):
    os.environ.pop(k, None)
os.environ['HUGGINGFACE_HUB_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
# 兼容旧/new huggingface_hub API（可选）
try:
    import huggingface_hub as _hf
    if not hasattr(_hf, "cached_download") and hasattr(_hf, "hf_hub_download"):
        _hf.cached_download = _hf.hf_hub_download
    if not hasattr(_hf, "cached_path") and hasattr(_hf, "hf_hub_download"):
        _hf.cached_path = _hf.hf_hub_download
except Exception:
    pass
# ...existing code...

# 在代码开始处添加
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# 加载预训练模型权重（被注释掉的代码行）
# pipe.diffusion_prior.load_state_dict(torch.load(f'./fintune_ckpts/{config['data_path']}/{sub}/{model_name}.pt', map_location=device))
save_path = f'./fintune_ckpts/{config["encoder_type"]}/{sub}notebook_st/{model_name}notebook_st.pt'
# 构建模型保存路径
# 获取路径中的目录部分
directory = os.path.dirname(save_path)

# Create the directory if it doesn't exist
# 如果目录不存在则创建目录，exist_ok=True表示如果目录已存在不会抛出异常
os.makedirs(directory, exist_ok=True)


from PIL import Image
# 导入os模块，用于操作系统相关功能
import os


generator = Generator4Embeds(num_inference_steps=4, device=device)
  
# 定义生成图像的保存目录 
directory = f"generated_imgs/{sub}notebook_Org"
# 循环200次，处理每个EEG嵌入 生成图像 这里的200对应测试集的200个样本 sdxl没有任何修改直接当成工具进行使用

for k in range(200):
    # 提取第k个EEG嵌入，保持维度使用k:k+1
    eeg_embeds = emb_eeg_test[k:k+1]
    # 使用管道生成图像，传入条件嵌入、推理步数和指导比例
    h = pipe.generate(c_embeds=eeg_embeds, num_inference_steps=50, guidance_scale=5.0)
    
    # 对每个EEG嵌入生成10个不同的图像
    for j in range(10):
         # 生成图像，将隐藏状态转换为float16精度
        image = generator.generate(h.to(dtype=torch.float16))
        # Construct the save path for each image
        # 构建每个图像的保存路径
        path = f'{directory}/{texts[k]}/{j}.png'
        # Ensure the directory exists

        # 确保图像保存的目录存在，如果不存在则创建
        os.makedirs(os.path.dirname(path), exist_ok=True)
        # Save the PIL Image
        # 保存PIL图像到指定路径
        image.save(path)
        # 打印图像保存的路径信息
        print(f'Image saved to {path}')